# Dot product through a basis: $a \cdot b = b \cdot a$

This notebook derives the commutativity of the dot product from first
principles, using the **basis bridge** (`tender.basis`): expand the
invariant vectors in a coordinate frame, turn the basis dot products into
Kronecker deltas, and contract.

In [ ]:
import tender
import tender.basis as tb
import tender.derivation as td

## 1. A coordinate frame and two vectors

`tb.wcs` returns the orthonormal World Cartesian frame $i, j, k$.

In [ ]:
ctx = tender.Context()
frame = tb.wcs(ctx)
a = tender.tensor("a", rank=1, ctx=ctx)
b = tender.tensor("b", rank=1, ctx=ctx)
ab = a @ b   # @ is the dot product
ab

## 2. Expand each vector in the basis

$a = a^i e_i$, $b = b^j e_j$ (covariant expansion; orthonormal, so the
coordinates are written with lower indices).

In [ ]:
expanded = tb.expand_in_basis(ab, frame, tb.Variance.Covariant)
expanded

## 3. Basis dots become Kronecker deltas

`simplify_basis_dot` rewrites $e_i \cdot e_j \to \delta_{ij}$ and pulls
the scalar coordinates out of the contraction; `canonicalize` tidies the
result and materializes the implicit Einstein sums.

In [ ]:
with_delta = td.canonicalize(tb.simplify_basis_dot(expanded, frame))
with_delta

## 4. Contract the deltas

Evaluate the deltas concretely and re-fold the result into a single sum.
The dot product reduces to the scalar coordinate contraction
$\sum_i a_i b_i$.

In [ ]:
def contract(expr):
    expr = td.unroll_sums(expr)
    expr = td.eval_delta_concrete(expr)
    expr = td.fold_arithmetic(expr)
    return td.fold_sums(td.canonicalize(expr))

ab_reduced = contract(with_delta)
ab_reduced

## 5. $b \cdot a$ reduces to the same thing

Running the identical pipeline on $b \cdot a$ gives the same scalar sum —
the coordinates are scalars, and the canonicalizer commutes them. Hence
$a \cdot b = b \cdot a$.

In [ ]:
def reduce_dot(expr):
    expr = tb.expand_in_basis(expr, frame, tb.Variance.Covariant)
    expr = tb.simplify_basis_dot(expr, frame)
    return contract(td.canonicalize(expr))

ba_reduced = reduce_dot(b @ a)
print("a.b =", ab_reduced.latex())
print("b.a =", ba_reduced.latex())
td.algebraic_eq(ab_reduced, ba_reduced)

## 6. Round trip: expand then fold back

`reassemble` is the inverse of `expand_in_basis`: it recognizes a
coordinate expansion in the frame and folds it back to the invariant.
Here a rank-2 tensor $A$ makes the round trip exactly.

In [ ]:
A = tender.tensor("A", rank=2, ctx=ctx)
expanded_A = td.canonicalize(tb.expand_in_basis(A, frame, tb.Variance.Covariant))
back = tb.reassemble(expanded_A, frame)
print("expanded:", expanded_A.latex())
td.structural_eq(back, A)